# Blacknaught M2: how long will this power cut last?
Trained **on this IBM LinuxONE z17 VM** from 306,251 real Electricity North West outages (CC BY 4.0), tested on ENWL 2024–26 and on 178,864 UK Power Networks outages the models never saw. Every model gives S(t) = P(power still off t hours after the fault started); triage compares it with each patient's backup.

In [1]:
import platform, subprocess, sys
print('machine:', platform.machine(), '| byte order:', sys.byteorder)
print(subprocess.run("lscpu | grep -E 'Machine type|Hypervisor'", shell=True, capture_output=True, text=True).stdout)
print('NNPA facility:', 'yes' if 'nnpa' in open('/proc/cpuinfo').read().split() else 'no')

machine: s390x | byte order: big
Machine type:                            9175
Hypervisor:                              z/VM 7.3.0
Hypervisor vendor:                       IBM

NNPA facility: yes


In [2]:
import os, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
sys.path.insert(0, '/home/ibm-user/shared/nate')
from pathlib import Path
import numpy as np, pandas as pd
from ml import m2
enwl, ukpn = m2.load(Path('/home/ibm-user/shared/nate/data/prepared'))
train, valid, test = (enwl[enwl['split'] == s] for s in ('train', 'validate', 'test'))
median = float(train['customers'].median())
rng = np.random.default_rng(m2.SEED)
print(f'train {len(train):,} (2000-22) | validate {len(valid):,} (2023) | test {len(test):,} (2024-26) | UKPN {len(ukpn):,}')
print('features:', list(m2.features(train.head(), median).columns))

train 269,170 (2000-22) | validate 9,828 (2023) | test 27,253 (2024-26) | UKPN 178,864
features: ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'weekend', 'log_customers', 'volt_lv', 'volt_hv', 'volt_unknown']


## 1. M1 baselines
Kaplan–Meier (the live stand-in), Kaplan–Meier per cause (cause isn't known live, so this one is offline only), and a log-linear model on the live features.

In [3]:
baselines = []
for build in (lambda: m2.Km(train['duration_h']), lambda: m2.KmByCause(train), lambda: m2.LogNormal(train, median)):
    t0 = time.perf_counter(); model = build(); baselines.append(model)
    print(f'{model.name}: trained in {time.perf_counter() - t0:.1f} s')

M1 Kaplan-Meier: trained in 0.0 s
M1 Kaplan-Meier per cause: trained in 0.1 s


M1 log-linear: trained in 0.2 s


## 2. M2 XGBoost AFT
Gradient-boosted accelerated-failure-time model. Voltage is hidden on a third of training rows so outages with no voltage (all of UKPN) get a trained answer.

In [4]:
t0 = time.perf_counter()
aft = m2.Aft(train, valid, median, rng)
print(f'trained in {time.perf_counter() - t0:.1f} s, best round {aft.booster.best_iteration}, spread sigma {aft.sigma:.3f}')

[0]	validate-aft-nloglik:4.49977


[73]	validate-aft-nloglik:2.85265


trained in 1.5 s, best round 48, spread sigma 1.228


## 3. M2 Keras discrete-time hazard MLP
24 hourly bins plus one for longer outages. This is the network that IBM's zDNN plugin runs on the Telum NNPA.

In [5]:
t0 = time.perf_counter()
mlp = m2.HazardMlp(train, valid, median, rng)
print(f'trained in {time.perf_counter() - t0:.1f} s')

Epoch 1/12


66/66 - 1s - 18ms/step - loss: 2.7571 - val_loss: 2.4741


Epoch 2/12


66/66 - 0s - 5ms/step - loss: 2.5767 - val_loss: 2.4741


Epoch 3/12


66/66 - 0s - 4ms/step - loss: 2.5557 - val_loss: 2.4741


trained in 5.0 s


## 4. Scores on outages none of the models trained on
Lower Brier and integrated Brier (IBS) are better; C-index 0.5 means no ranking skill. KM is the model triage uses today.

In [6]:
models = baselines + [aft, mlp]
board = m2.scoreboard(models, {'ENWL 2024-26': test, 'UKPN cross-check': ukpn})
km = board[board['model'] == 'M1 Kaplan-Meier'].set_index('test')['ibs_24h']
board['ibs_skill_vs_km'] = (1 - board['ibs_24h'] / board['test'].map(km)).round(3)
board

,model,test,brier_2h,brier_4h,brier_8h,ibs_24h,c_index,n,ibs_skill_vs_km
0,M1 Kaplan-Meier,ENWL 2024-26,0.2334,0.2460,0.1708,0.1210,0.5000,27253,0.000
1,M1 Kaplan-Meier,UKPN cross-check,0.2438,0.2442,0.1809,0.1383,0.5000,178864,0.000
2,M1 Kaplan-Meier per cause,ENWL 2024-26,0.1926,0.2067,0.1546,0.1092,0.6385,27253,0.098
3,M1 Kaplan-Meier per cause,UKPN cross-check,0.2438,0.2442,0.1809,0.1383,0.5000,178864,0.000
4,M1 log-linear,ENWL 2024-26,0.2383,0.2496,0.1741,0.1220,0.5323,27253,-0.008
5,M1 log-linear,UKPN cross-check,0.2420,0.2436,0.1821,0.1388,0.4938,178864,-0.004
6,M2 XGBoost AFT,ENWL 2024-26,0.2234,0.2367,0.1654,0.1144,0.5958,27253,0.055
7,M2 XGBoost AFT,UKPN cross-check,0.2604,0.2601,0.1961,0.1486,0.5172,178864,-0.074
8,M2 Keras hazard MLP,ENWL 2024-26,0.2303,0.2447,0.1674,0.1168,0.5346,27253,0.035
9,M2 Keras hazard MLP,UKPN cross-check,0.2461,0.2453,0.1835,0.1407,0.5064,178864,-0.017


In [7]:
out = Path('/home/ibm-user/shared/nate-runs/models'); out.mkdir(parents=True, exist_ok=True)
aft.booster.save_model(out / 'm2_aft.json'); mlp.model.save(out / 'm2_hazard.keras')
import json
(out / 'm2_metrics.json').write_text(json.dumps({'customers_median': median, 'aft_sigma': aft.sigma,
    'scores': board.to_dict(orient='records')}, indent=1))
print(sorted(p.name for p in out.iterdir()))

['m2_aft.json', 'm2_hazard.keras', 'm2_metrics.json']
